In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
from dotenv import load_dotenv
load_dotenv()
open_api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=open_api_key)

In [2]:
client = OpenAI()

In [ ]:
instructions = """You are the exit advisor for an SMS recruiting chatbot for a Python developer position.
Your job is to decide whether the conversation should end now. If it should end, provide a short closing SMS for the candidate.
* Read the full conversation before deciding.
* End the conversation when the candidate is clearly not interested, asks to stop contact, has accepted another job, is the wrong person,the candidate has no programming experience at all, or the thread is naturally closed with a clear goodbye.
* Do not end the conversation on weak signals alone (e.g. a brief "thanks" early on, or a neutral question).
* If not ending, another advisor will handle the reply — do not write a candidate message.
* Respond in exactly this format (no extra text): 

If ending:
end_conversation: yes
message: <short polite SMS closing>

If not ending:
end_conversation: no

If ending:
end_conversation: yes
message: <short polite SMS closing>

If not ending:
end_conversation: no"""


text = "I've been using Python professionally for five years, mostly for data analysis."


completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": text}
    ]
)


print(completion.choices[0].message.content)

end_conversation: no


In [4]:
instructions = """You are the exit advisor for an SMS recruiting chatbot for a Python developer position.
Your job is to decide whether the conversation should end now. If it should end, provide a short closing SMS for the candidate.
* Read the full conversation before deciding.
* End the conversation when the candidate is clearly not interested, asks to stop contact, has accepted another job, is the wrong person, or the thread is naturally closed with a clear goodbye.
* Do not end the conversation on weak signals alone (e.g. a brief "thanks" early on, or a neutral question).
* If not ending, another advisor will handle the reply — do not write a candidate message.
* Respond in exactly this format (no extra text): 

If ending:
end_conversation: yes
message: <short polite SMS closing>

If not ending:
end_conversation: no

If ending:
end_conversation: yes
message: <short polite SMS closing>
3
If not ending:
end_conversation: no"""


text = "i dont know what is even a string, sorry."


completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": text}
    ]
)


print(completion.choices[0].message.content)

end_conversation: yes
message: No worries at all! If you're ever interested in learning more about programming roles in the future, feel free to reach out. Have a great day!


In [ ]:
eval_obj = client.evals.create(
    name="Exit Advisor Decision Routing",

    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "conversation_id": {"type": "integer"},
                "turn_id": {"type": "integer"},
                "model_input": {"type": "string"},
                "label": {"type": "string"},
            },
            "required": [
                "conversation_id",
                "turn_id",
                "model_input",
                "label"
            ],
        },
        "include_sample_schema": True,
    },

    testing_criteria=[
        {
            "type": "string_check",
            "name": "Match exit decision to human label",
            "input": "{{ sample.output_text }}",
            "operation": "eq",
            "reference": "{{ item.label }}",
        }
    ],
)

print(eval_obj)
print(eval_obj.id)




EvalCreateResponse(id='eval_6a52564703348191aaccbc25789fe6ce', created_at=1783780935, data_source_config=EvalCustomDataSourceConfig(schema_={'type': 'object', 'properties': {'item': {'type': 'object', 'properties': {'conversation_id': {'type': 'integer'}, 'turn_id': {'type': 'integer'}, 'model_input': {'type': 'string'}, 'label': {'type': 'string'}}, 'required': ['conversation_id', 'turn_id', 'model_input', 'label']}, 'sample': {'type': 'object', 'properties': {'model': {'type': 'string'}, 'choices': {'type': 'array', 'items': {'type': 'object', 'properties': {'message': {'type': 'object', 'properties': {'role': {'type': 'string', 'enum': ['assistant']}, 'content': {'type': ['string', 'array', 'null']}, 'refusal': {'type': ['boolean', 'null']}, 'tool_calls': {'type': ['array', 'null'], 'items': {'type': 'object', 'properties': {'type': {'type': 'string', 'enum': ['function']}, 'function': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'arguments': {'type': 'string'}}, 'r

In [6]:
file = client.files.create(
    file=open("sms_conversations.jsonl", "rb"),
    purpose="evals"
)


print(file)
print(file.id)


FileObject(id='file-CWeTzvHCqqhD2ba28y8NCD', bytes=26645, created_at=1783780935, filename='sms_conversations_final.jsonl', object='file', purpose='evals', status='processed', expires_at=None, status_details=None)
file-CWeTzvHCqqhD2ba28y8NCD


In [7]:
print(eval_obj.id)
print(file.id)

eval_6a52564703348191aaccbc25789fe6ce
file-CWeTzvHCqqhD2ba28y8NCD


In [ ]:
instructions = """
You are an expert recruiter reviewing SMS conversations with job candidates.

You will be given a complete conversation between a recruiter and a candidate.

Your task is to determine whether the conversation should end at its current state.

Read the entire conversation before making your decision.

If the conversation has reached a natural conclusion (for example, an interview has been scheduled, the candidate is no longer interested, the candidate requested to stop receiving messages, or there is nothing meaningful left to discuss), respond with:

end

Otherwise, if the recruiter should continue the conversation to gather more information, answer questions, or continue scheduling, respond with:

continue

Respond with only one word:
end
or
continue
"""

run = client.evals.runs.create(
    eval_obj.id,  # YOUR_EVAL_ID

    name="Exit Advisor Decision Routing Run",

    data_source={
        "type": "completions",
        "model": "gpt-4.1",

        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "developer",
                    "content": instructions
                },
                {
                    "role": "user",
                    "content": "{{ item.model_input }}"
                }
            ],
        },

        "source": {
            "type": "file_id",
            "id": file.id  # your Stage 2 JSONL file
        },
    },
)
print(run)
print(run.id)

RunCreateResponse(id='evalrun_6a52564c117c819190a264c3d96c3b68', created_at=1783780940, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-CWeTzvHCqqhD2ba28y8NCD', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='\nYou are an expert recruiter reviewing SMS conversations with job candidates.\n\nYou will be given a complete conversation between a recruiter and a candidate.\n\nYour task is to determine whether the conversation should end at its current state.\n\nRead the entire conversation before making your decision.\n\nIf the conversation has reached a natural conclusion (for example, an interview has been scheduled, the candidate is no longer interested, the candidate requested to stop receiving messages, or there is nothing meaningful left to discuss), respond with:\n\nend\n\nOtherwise, if the recruiter should continue the conversation to gather more information, answer questions, or continue schedu

In [ ]:
run_retrieve = client.evals.runs.retrieve(
    eval_id=eval_obj.id, # YOUR_EVAL_ID
    run_id=run.id # YOUR_RUN_ID
    )


print(run_retrieve)
print(run_retrieve.status)

RunRetrieveResponse(id='evalrun_6a52564c117c819190a264c3d96c3b68', created_at=1783780940, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-CWeTzvHCqqhD2ba28y8NCD', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='\nYou are an expert recruiter reviewing SMS conversations with job candidates.\n\nYou will be given a complete conversation between a recruiter and a candidate.\n\nYour task is to determine whether the conversation should end at its current state.\n\nRead the entire conversation before making your decision.\n\nIf the conversation has reached a natural conclusion (for example, an interview has been scheduled, the candidate is no longer interested, the candidate requested to stop receiving messages, or there is nothing meaningful left to discuss), respond with:\n\nend\n\nOtherwise, if the recruiter should continue the conversation to gather more information, answer questions, or continue sche

In [28]:
run_retrieve.result_counts

ResultCounts(errored=0, failed=19, passed=40, total=59)

In [ ]:
with open("few_shot_prompt_trial.txt", "r", encoding="utf-8") as f:
    new_instructions = f.read()


In [25]:
passed = run_retrieve.result_counts.passed
total = run_retrieve.result_counts.total

print('Accuracy')
print(len('Accuracy')*'-')
print(f'{(passed / total) :.2%}')

Accuracy
--------


ZeroDivisionError: division by zero

In [26]:
run_retrieve.report_url

'https://platform.openai.com/evaluations/eval_6a52564703348191aaccbc25789fe6ce?project_id=proj_fbhZpTHb89C5f2WcCBGZrDzS&run_id=evalrun_6a52564c117c819190a264c3d96c3b68'

In [23]:
items = client.evals.runs.output_items.list(
    eval_id=eval_obj.id, # YOUR_EVAL_ID
    run_id=run.id # YOUR_RUN_ID
    )


items_list = list(items)
items_list[:5]

[OutputItemListResponse(id='outputitem_6a525652285881919b12ac2ecd200e84', created_at=1783780946, datasource_item={'conversation_id': 15, 'turn_id': 5, 'model_input': "Recruiter: Hi, thanks for submitting your application for our Python Developer role. Could you share a bit about your Python experience?\nCandidate: I've been using Python professionally for five years, mostly for ML.\nRecruiter: Could we schedule a chat this Friday at 11\u202fAM or next Monday at 9\u202fAM?\nCandidate: I'm sorry, but I'm no longer interested.\nRecruiter: Understood. I'll close your application for now. Best of luck in your job search!", 'label': 'end'}, datasource_item_id=58, eval_id='eval_6a52564703348191aaccbc25789fe6ce', object='eval.run.output_item', results=[Result(name='Match exit decision to human label-0f19200e-47ad-4d31-840f-9f1f99a22b88', passed=True, score=1.0, sample=None, type=None)], run_id='evalrun_6a52564c117c819190a264c3d96c3b68', sample=Sample(error=None, finish_reason='stop', input=[Sa

In [ ]:
with open("few_shot_prompt_exit_advisior.txt", "r", encoding="utf-8") as f:
    updated_instructions = f.read()

text = "I've been using Python professionally for five years, mostly for data analysis."

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": text}
    ]
)


print(completion.choices[0].message.content)



continue


In [ ]:
with open("few_shot_prompt_exit_advisior.txt", "r", encoding="utf-8") as f:
    updated_instructions = f.read()

text = "I've been using Python professionally for five years, mostly for data analysis."

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": text}
    ]
)


print(completion.choices[0].message.content)


In [40]:
with open("few_shot_prompt_exit_advisior.txt", "r", encoding="utf-8") as f:
    updated_instructions = f.read()

text = "I've been using Python for a couple of month, I don't have much experience in that area, but I'm eager to learn and quickly adapt."

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": text}
    ]
)


print(completion.choices[0].message.content)




continue


In [43]:
# few shot prompt eng 

with open("few_shot_prompt_trial.txt", "r", encoding="utf-8") as f:
    trial_instructions = f.read()

trial_run = client.evals.runs.create(
    eval_id=eval_obj.id,

    name="Exit Advisor Few-Shot Trial Run",

    data_source={
        "type": "completions",
        "model": "gpt-4.1",

        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "developer",
                    "content": trial_instructions
                },
                {
                    "role": "user",
                    "content": "{{ item.model_input }}"
                }
            ],
        },

        "source": {
            "type": "file_id",
            "id": file.id  # אותו קובץ JSONL שכבר העלית
        },
    },
)

print(trial_run.id)

evalrun_6a551c8606dc8191bc43f03258766d24


In [42]:
run_retrieve = client.evals.runs.retrieve(
    eval_id=eval_obj.id,
    run_id="evalrun_6a551b8baafc8191909ba4aaebc76f1d"
)

print(run_retrieve)
print(run_retrieve.status)

RunRetrieveResponse(id='evalrun_6a551b8baafc8191909ba4aaebc76f1d', created_at=1783962508, data_source=CreateEvalCompletionsRunDataSource(source=SourceFileID(id='file-CWeTzvHCqqhD2ba28y8NCD', type='file_id'), type='completions', input_messages=InputMessagesTemplate(template=[EasyInputMessage(content='\n# Identity\n\nYou are the exit advisor for an SMS recruiting chatbot for a Python developer position.\nYour job is to decide whether the conversation should end now. If it should end, provide a short closing SMS for the candidate.\n\n# Instructions\n\n* Read the full conversation before deciding.\n* End the conversation when the candidate is clearly not interested, asks to stop contact, has accepted another job, is the wrong person, or the thread is naturally closed with a clear goodbye.\n* Do not end the conversation on weak signals alone (e.g. a brief "thanks" early on, or a neutral question).\n* If not ending, another advisor will handle the reply — do not write a candidate message.\n*